### Imports

In [1]:
import pandas as pd
from IPython.display import display,clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scipy.stats as stats
import numpy as np
import ipywidgets as widgets
import statsmodels.api as sm
import sys, os
import ipywidgets as widgets


#Import des modules 
sys.path.append(os.path.abspath("src/Modules"))
import exploration.exploration as exploration
import exploration.clean_fusion as clean_fusion
import features.new_features as new_features
import features.features_selection as features_selection
import models__analyses.train_test as train_test


import models__analyses.model as model
import models__analyses.etude_model as etude_model

import optimisations__analyses.shap_analyse as shap_analyse
import optimisations__analyses.hyperparameters_interface as hyperparameters_interface

#Import librairies pour les modules
import inspect
from matplotlib.lines import Line2D
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix
from IPython.display import display, clear_output


In [2]:
df_eval = pd.read_csv("extrait_eval.csv")
df_sirh = pd.read_csv("extrait_sirh.csv")
df_sondage = pd.read_csv("extrait_sondage.csv")

# 1. Analyse exploratoire

In [3]:
from exploration.exploration import explorer, analyser_attrition_globale, bouton_explorer
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Bouton Exploration Globale
bouton = bouton_explorer()
sortie = widgets.Output()

def au_clic(_):
    with sortie:
        clear_output(wait=True)
        explorer(df_sirh, df_sondage, df_eval)

bouton.on_click(au_clic)

display(widgets.VBox([widgets.Label("→ 1. Exploration Standard"), bouton, sortie]))


# 2. Bouton Analyse Globale de l'Attrition

#Création du bouton
bouton_attrition_global = widgets.Button(
    description="Analyser l'Attrition Globale",
    button_style="success",
    icon="globe",
    layout=widgets.Layout(width="280px", height="40px"),
)
sortie_attrition_global = widgets.Output()
#clear outpout pour relancer
def au_clic_attrition_global(_):
    with sortie_attrition_global:
        clear_output(wait=True)
        analyser_attrition_globale(df_sirh, df_sondage, df_eval)

bouton_attrition_global.on_click(au_clic_attrition_global)
display(widgets.VBox([widgets.Label("→ 2. Analyse Avancée de l'Attrition"), bouton_attrition_global, sortie_attrition_global]))

# 1.1 Hypothèse



#### <div style="font-size:18px;line-height:1.55;color:#1c2333;background:#fffdf9;border:1px solid #e6dfd4;border-radius:18px;padding:28px 32px;font-family:Georgia,'Iowan Old Style',Palatino,serif;">

<p style="margin:0 0 8px;font-family:'Segoe UI',system-ui,sans-serif;font-size:12px;letter-spacing:.16em;text-transform:uppercase;color:#c45c2a;font-weight:700;">
Hypothèse · Attrition
</p>

<h3 style="margin:0 0 16px;font-size:26px;line-height:1.25;font-weight:600;">
Un profil junior peu ancré, mais attention ce n'est pas une cause unique.
</h3>

<p style="margin:0 0 14px;">
L’attrition ne se répartit pas au hasard : elle se concentre chez les salariés
<strong>en début de parcours</strong> → plus jeunes, niveau de poste bas, salaire plus faible,
peu d’ancienneté dans l’entreprise, dans le poste et sous le responsable actuel.
</p>

<p style="margin:0 0 14px;">
Les matrices Pearson et Spearman le montrent directement : âge, revenu, expérience totale,
ancienneté, années dans le poste / sous le responsable et niveau hiérarchique forment un
<strong>bloc rouge</strong>. Elles varient ensemble. On ne peut donc pas dire
« c’est le salaire » ou « c’est l’âge » : la matrice dit que c’est <em>le même axe</em>.
</p>

<p style="margin:0 0 14px;">
La ligne <strong>a_quitte_l_entreprise</strong> de ces mêmes matrices reste claire :
le départ est associé à ce bloc, les vraiables allant de environ -17 à envrion −0,20</span>.<br>
Les satisfactions sont à l’écart de ce bloc (cases pâles) : et nous pourront croiser ces signaux avec des des indicateurs fortement corrélés à la cible, <br>
notament car les causes de l'attrition sont multicausales et que dans le cadre d'une recherche de causes qualitatives, ces indicateurs unifiés produirait un biais et empecherait la variance.<br>

Les variables qualitatives telques<strong>+ 0,24</strong> les heures supplémentaires et le poste également a envrion + 0,24 ou le sstatus marital a +0,17 sont des atouts <em>qualitatifs</em> 
 par opposition à l'age, la rémunération, le nombre d'année dans le poste, participation pee ou nombre d'année dans l'entreprise.
</p>

<p style="margin:0 0 14px;">
Le diplôme, le genre et la date de dernière promotion on une influence quasiment null.
</p>

<p style="margin:0;padding-top:14px;border-top:1px solid #e6dfd4;color:black;font-size:16px;">
Hypothèse : un <strong style="color:#1c2333;">profil junior peu ancré</strong>, lu dans les
<strong style="color:#1c2333;">bloc de corrélations, les courbe KDE et les dérivées des indicateurs vers y</strong> → sont plusieurs facteurs liés entre eux,
mais pas une cause unique.
</p>

</div>

# 2 Nettoyage et fusion des fichiers

In [4]:
def get_sources():
    return df_sirh, df_sondage, df_eval

def save_clean(res):
    global df_clean
    df_clean = res

# Affichage du bouton interactif
clean_fusion.bouton_nettoyer_fusionner(get_sources, save_clean)

# 4 Créer les features

In [5]:
def get_df():
    return df_clean

def save_df(res):
    global df_clean, X, y
    df_clean = res
    X, y = new_features.preparer_X_y(df_clean)
    assert X is not None and y is not None, "❌ Erreur : X ou y n'a pas pu être créé."
    assert len(X) == len(y), f"❌ Erreur de dimension : X a {len(X)} lignes mais y a {len(y)} lignes."
    assert X.shape[0] > 0, "❌ Erreur : Le DataFrame X est vide."
    X.to_csv("X.csv", index=False)
    y.to_csv("y.csv", index=False)
    print(f"✅ Validation réussie : X ({X.shape[1]} features) et y sont prêts, synchronisés et sauvegardés !")

display(new_features.bouton_creer_features(get_df, save_df))

In [6]:
def get_current_df():
    return df_clean

def set_current_df(new_df):
    global df_clean
    df_clean = new_df

exploration.bouton_analyse_features(get_current_df, set_current_df)

<div style="font-size:20px;line-height:1.65;color:#1c2333;background:#fffdf9;border:1px solid #e6dfd4;border-radius:16px;padding:32px 36px;font-family:Georgia,Palatino,serif;">

<p style="margin:0 0 8px;font-family:'Segoe UI',sans-serif;font-size:14px;letter-spacing:.14em;text-transform:uppercase;color:#c45c2a;font-weight:700;">
Forme du lien avec le départ
</p>
<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">1. On a testé, puis classé : 4 groupes observés</p>


<p style="margin:0 0 16px;">
<strong>Quantitatives linéaires (14)</strong>&nbsp; <code>nombre_experiences_precedentes</code>, <code>nb_formations_suivies</code>, <code>distance_domicile_travail</code>, <code>niveau_education</code>, <code>annees_depuis_la_derniere_promotion</code>, <code>note_evaluation_precedente</code>, <code>satisfaction_employee_nature_travail</code>, <code>satisfaction_employee_equipe</code>, <code>augmentation_salaire_precedente</code>, <code>montant_augmentation_precedente</code>, <code>revenu_par_niveau</code>, <code>satisfaction_globale</code>, <code>satisfaction_min</code>, <code>delta_performance</code>.
</p>

<p style="margin:0 0 16px;">
<strong>Quantitatives non-linéaires (14)</strong> → courbe confirmée, β₂ ≠ 0 : <code>age</code>, <code>revenu_mensuel</code>, <code>annee_experience_totale</code>, <code>annees_dans_l_entreprise</code>, <code>annees_dans_le_poste_actuel</code>, <code>nombre_participation_pee</code>, <code>annes_sous_responsable_actuel</code>, <code>satisfaction_employee_environnement</code>, <code>niveau_hierarchique_poste</code>, <code>satisfaction_employee_equilibre_pro_perso</code>, <code>ratio_anciennete_carriere</code>, <code>inertie_poste</code>, <code>stagnation_promotion</code>, <code>revenu_par_annee_experience</code>.
</p>

<p style="margin:0 0 16px;">
<strong>Catégorielles (7)</strong> → une catégorie par valeur : <code>genre</code>, <code>statut_marital</code>, <code>departement</code>, <code>poste</code>, <code>domaine_etude</code>, <code>frequence_deplacement</code>, <code>heure_supplementaires</code>.
</p>

<p style="margin:0 0 16px;">
<strong>Booléennes (4)</strong> → oui/non, pas de courbe possible : <code>hs_et_salaire_bas</code>, <code>jeune_faible_anciennete</code>, <code>trajet_long</code>, <code>job_hopper</code>.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">2. Les p-values (échantillonage statistique) confirment que ce n'est pas du hasard</p>

<p style="margin:0 0 16px;">
Pour le groupe « non-linéaires », plus la p-value du terme x² est petite, plus la courbure est certaine (pas un effet de bruit). Les plus nettes : <code>annees_dans_l_entreprise</code> (p = 3,99e-09), <code>nombre_participation_pee</code> (p = 1,29e-08), <code>annee_experience_totale</code> (p = 3,37e-07), <code>age</code> (p = 1,37e-06). Toutes les variables du groupe restent sous le seuil de 0,05 → la courbure est donc statistiquement confirmée pour chacune, pas juste suggérée par le graphique.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">3. Les KDE confirment à l'œil</p>

<p style="margin:0 0 16px;">
Même liste de variables non-linéaires : le récit junior s'y lit directement car <strong>les départs sont plus présents lorsque la courbe orange est plus grande</strong> (le pic orange domine nettement à gauche de l'axe, traduisant une surreprésentation des départs sur les valeurs faibles). C'est particulièrement net sur <code>age</code>, <code>revenu_mensuel</code>, <code>annee_experience_totale</code>, les trois anciennetés (<code>annees_dans_l_entreprise</code>, <code>annees_dans_le_poste_actuel</code>, <code>annes_sous_responsable_actuel</code>), <code>inertie_poste</code>, <code>niveau_hierarchique_poste</code> et <code>nombre_participation_pee</code> (absence de PEE à 0).<br>
À l'inverse, les deux satisfactions sont non-linéaires au test, mais visuellement plus bosselées (notes 1–4) qu'un vrai décalage global. Là où les courbes se superposent, il y a peu de signal, quel que soit β₂.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">4. Les matrices referment la boucle</p>

<p style="margin:0 0 16px;">
Pearson et Spearman montrent que les non-linéaires <strong>bloc carrière</strong> ne sont pas des signaux séparés →&nbsp; c'est
<strong>un bloc rouge</strong> : <code>age</code>, <code>revenu_mensuel</code>, <code>annee_experience_totale</code>, <code>annees_dans_l_entreprise</code>, <code>annees_dans_le_poste_actuel</code>, <code>annes_sous_responsable_actuel</code>, <code>niveau_hierarchique_poste</code>.
Spearman un peu plus élevé = ensemble monotone → cohérent avec β₂ ≠ 0 (exemple note par echellons successifs, classement rang par rang).
</p>

<p style="margin:0 0 16px;">
Lien de ce bloc avec <strong>a_quitte_l_entreprise</strong> (y, le départ) : <strong>corrélation modérée</strong> (environ −0,20). β₂ + p-values + KDE + matrices racontent <em>une seule hypothèse</em> et pas sept causes indépendantes et il faut ajouter a la cause des attritions le fait que les satisfactions restent des variables Likert, structurellement différentes des autres.
</p>

<p style="margin:0;padding-top:16px;border-top:1px solid #e6dfd4;color:#5b6578;font-size:18px;">
Chaîne de preuve : le test x² classe chaque variable en linéaire ou non-linéaire → les p-values disent si c'est fiable → le KDE montre où ça se voit à l'œil (quand la courbe orange dépasse la bleue) → les matrices montrent que c'est un seul bloc cohérent, pas plusieurs histoires séparées. Les 4 groupes (linéaires, non-linéaires, catégorielles, booléennes) sont donc une classification <em>observée</em>, pas décidée à l'avance.
</p>

</div>

# Selectionner les Features 

In [7]:

def get_df():
    return df_clean

def au_demarrage(variables_choisies):
    global variables_selectionnees
    variables_selectionnees = variables_choisies
    print("Prêt pour la suite avec :", variables_choisies)

def au_panneau_pret(vb):
    global var_buttons
    var_buttons = vb

display(features_selection.bouton_features_selection(
    get_df,
    au_demarrage,
    target_col="a_quitte_l_entreprise",
    on_panel_ready_callback=au_panneau_pret,
))

# 5 Train/Test

In [8]:
from sklearn.model_selection import StratifiedKFold

def get_df():
    return df_clean

def get_var_buttons():
    if 'var_buttons' in globals():
        return var_buttons
    print("⚠️ Le panneau de sélection des features n'a pas été lancé !")
    return None

def on_split_termine(resultats):
    global X_train, X_test, y_train, y_test, features_selectionnees, cv
    X_train = resultats["X_train"]
    X_test = resultats["X_test"]
    y_train = resultats["y_train"]
    y_test = resultats["y_test"]
    features_selectionnees = resultats["features_selectionnees"]
    
    # Récupération dynamique de la graine choisie dans l'interface de split
    seed_dynamique = resultats.get("random_state", 42)
    
    # Création du CV global dynamique pour l'interface d'hyperparamètres
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed_dynamique)

display(train_test.bouton_train_test(get_df, get_var_buttons, on_split_termine, target_col="a_quitte_l_entreprise"))

# 6 Préprocesseur 

In [9]:
# ============================================================
# ÉTAPE 2 — PRÉPROCESSEUR (fit sur le train seulement)
# ============================================================
btn_prep = widgets.Button(
    description="Lancer le préprocesseur",
    button_style="warning",
    layout=widgets.Layout(width="300px", height="40px"),
)
out_prep = widgets.Output()
display(widgets.VBox([btn_prep, out_prep]))

def on_prep(b):
    global preprocessor, X_train_prep, X_test_prep
    with out_prep:
        clear_output()
        if "X_train" not in globals():
            print("Lance d’abord la cellule 1 (train / test).")
            return
        if "var_buttons" not in globals():
            print("Le panneau de sélection des variables n'a pas été initialisé.")
            return
            
        # Récupération de la sélection active par groupe depuis le panneau interactif
        from features.features_selection import obtenir_variables_selectionnees
        
        selection_par_groupe = obtenir_variables_selectionnees(var_buttons, par_groupe=True)
        
        preprocessor = construire_preprocessor(
            colonnes_lineaires=selection_par_groupe.get("lineaires", []),
            colonnes_non_lineaires=selection_par_groupe.get("non_lineaires", []),
            cols_qualitatives=selection_par_groupe.get("qualitatives", []),
            cols_booleennes=selection_par_groupe.get("booleennes", []),
            cols_ratios=selection_par_groupe.get("nouvelles_features", []),
            inclure_lineaires=True,
            inclure_non_lineaires=True,
            inclure_qualitatives=True,
            inclure_booleennes=True,
            inclure_ratios=True,
            X=X_train,
        )
        
        preprocessor.fit(X_train)
        X_train_prep = preprocessor.transform(X_train)
        X_test_prep = preprocessor.transform(X_test)
        
        print("Fit uniquement sur X_train (pas de fuite du test).")
        print(f"X_train transformé : {getattr(X_train_prep, 'shape', type(X_train_prep))}")
        print(f"X_test transformé  : {getattr(X_test_prep, 'shape', type(X_test_prep))}")
        print("\nÉtape 2 OK → passe à la cellule modèles.")

btn_prep.on_click(on_prep)

<div style="font-size:20px;line-height:1.65;color:#1c2333;background:#fffdf9;border:1px solid #e6dfd4;border-radius:16px;padding:32px 36px;font-family:Georgia,Palatino,serif;">

<p style="margin:0 0 8px;font-family:'Segoe UI',sans-serif;font-size:14px;letter-spacing:.14em;text-transform:uppercase;color:#c45c2a;font-weight:700;">
Pipeline &amp; préparation des données
</p>

<h3 style="margin:0 0 20px;font-size:30px;line-height:1.25;font-weight:600;">
Préprocesseur, pipeline et cible asymétrique
</h3>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">1. Ordre des étapes</p>

<p style="margin:0 0 16px;background:#f4efe8;padding:14px 16px;border-radius:10px;">
<strong>1.</strong> Sélection des variables (panneau ON/OFF)<br>
<strong>2.</strong> Split train / test <strong>stratifié</strong> (80 / 20)<br>
<strong>3.</strong> Construction du préprocesseur<br>
<strong>4.</strong> Modèles évalués <strong>dans un <code>Pipeline</code> sklearn</strong>
</p>

<p style="margin:0 0 16px;">
Le préprocesseur n'est jamais fit sur le test, ni sur tout <code>df_clean</code>.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">2. Split observé</p>

<p style="margin:0 0 16px;">
<strong>Train :</strong> 1 176 lignes (16,2 % de départs)<br>
<strong>Test :</strong> 294 lignes (16,0 % de départs)<br>
<strong>40 variables</strong><br>
<code>random_state=42</code>, <code>stratify=y</code>
</p>

<p style="margin:0 0 16px;">
La cible <code>a_quitte_l_entreprise</code> est <strong>asymétrique</strong> (~16 % de départs).
On ne juge pas les modèles à l'accuracy seule (un modèle « toujours reste » ferait déjà ~84 %).
Métriques utiles : <strong>ROC-AUC</strong>, <strong>recall</strong>, <strong>F1</strong>, <strong>précision</strong>.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">3. Préprocesseur</p>

<table style="width:100%;border-collapse:collapse;margin:0 0 16px;font-size:18px;">
<thead>
<tr style="background:#f4efe8;">
<th style="text-align:left;padding:10px 12px;border:1px solid #e6dfd4;">Groupe</th>
<th style="text-align:left;padding:10px 12px;border:1px solid #e6dfd4;">Traitement</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding:10px 12px;border:1px solid #e6dfd4;">Numériques linéaires / non linéaires / ratios</td>
<td style="padding:10px 12px;border:1px solid #e6dfd4;"><code>StandardScaler</code></td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e6dfd4;">Qualitatives</td>
<td style="padding:10px 12px;border:1px solid #e6dfd4;"><code>OneHotEncoder(handle_unknown="ignore")</code></td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e6dfd4;">Booléennes (0/1)</td>
<td style="padding:10px 12px;border:1px solid #e6dfd4;"><code>passthrough</code></td>
</tr>
</tbody>
</table>

<p style="margin:0 0 12px;">
La cellule « Lancer le préprocesseur » sert de <strong>contrôle</strong>.
L'évaluation réelle se fait dans <code>evaluer_modeles</code> :
</p>

<pre style="margin:0;background:#f4efe8;padding:14px 16px;border-radius:10px;font-family:Consolas,Menlo,monospace;font-size:17px;overflow-x:auto;">Pipeline(
  preprocessor  →  classifieur
)</pre>

</div>

# Modelisation

In [10]:
from models__analyses.model import construire_preprocessor
#Ignorer les warnings de futures dépréciations de focntions scikit-learn 
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

display(model.interface_modelisation_etape())

# 7 Analyse des modèles

In [11]:
sys.path.append(os.path.abspath('src/Modules'))

display(etude_model.interface_etude_modeles_etape())

# Recherche d'hyper parametres 

In [12]:
import optimisations__analyses.hyperparameters_interface as hyperparameters_interface
from models__analyses.model import obtenir_modeles

dict_modeles_base = obtenir_modeles()

categories_modeles = {
    "Référence (Baseline)": ["Dummy_Stratified"],
    "Linéaires & Régularisés": [
        "LogisticRegression_None", "LogisticRegression_L1",
        "LogisticRegression_L2", "Ridge", "Lasso_LogReg", "ElasticNet_LogReg",
    ],
    "Arbres & Boosting": [
        "DecisionTree", "RandomForest", "GradientBoosting", "AdaBoost", "XGBoost",
    ],
    "Autres (SVM, KNN)": [
        "SVC_Prob", "SVC_RBF", "SVC_Linear", "SVC_Poly", "SVR_Linear", "KNN",
    ],
}

display(hyperparameters_interface.interface_tuning(dict_modeles_base, categories_modeles))

# Analyse Shap

In [13]:
sys.path.append(os.path.abspath('src/Modules'))

display(shap_analyse.interface_analyse_shap())